# Défi quotidien : incitations précises pour un contrôle et une qualité de production optimaux

**Scénario :** équipe Formation & Communication d'une multinationale tech. On transforme un paragraphe de politique interne (dense, formel, jargonneux) en contenu de micro-apprentissage clair pour une newsletter interne.

**Cycle de développement du prompt :** conception → génération → évaluation → amélioration.

Ce notebook couvre les cinq étapes :
1. Créer une invite contrôlant ton, format et longueur
2. Évaluer les résultats
3. Détecter et atténuer les hallucinations
4. Paraphrase approfondie (public : stagiaires juniors)
5. Variante d'extraction de citation

---

**Texte source (politique de sécurité d'accès à distance) :**

> « Les employés doivent s'assurer que tout accès distant aux systèmes internes est établi via le VPN sécurisé approuvé. En aucun cas, des connexions non sécurisées ou des appareils personnels dépourvus de protection des terminaux ne doivent être utilisés pour accéder à des données confidentielles ou à des communications sensibles. »

## (Optionnel) Configuration pour tester les prompts en direct

Les prompts ci-dessous sont **copiables tels quels** dans ChatGPT, Gemini ou DeepSeek. Pour les exécuter directement dans le notebook, vous pouvez utiliser l'API OpenAI (facultatif).

> Ajoutez votre clé dans les secrets Colab (🔑 à gauche) sous le nom `OPENAI_API_KEY`, puis décommentez le code.

In [ ]:
# !pip install openai --quiet
#
# from google.colab import userdata
# from openai import OpenAI
#
# client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
#
# def run_prompt(prompt, model="gpt-4o-mini", temperature=0.7):
#     """Envoie un prompt au modèle et renvoie la réponse texte."""
#     resp = client.chat.completions.create(
#         model=model,
#         messages=[{"role": "user", "content": prompt}],
#         temperature=temperature,
#     )
#     return resp.choices[0].message.content

In [ ]:
# Texte source réutilisé dans tout le notebook
TEXTE_SOURCE = (
    "Les employés doivent s'assurer que tout accès distant aux systèmes internes "
    "est établi via le VPN sécurisé approuvé. En aucun cas, des connexions non "
    "sécurisées ou des appareils personnels dépourvus de protection des terminaux "
    "ne doivent être utilisés pour accéder à des données confidentielles ou à des "
    "communications sensibles."
)

def compter_mots(texte):
    """Compte les mots d'un texte (utile pour la contrainte < 75 mots)."""
    return len(texte.split())

print("Texte source chargé —", compter_mots(TEXTE_SOURCE), "mots.")

## Étape 1 : Créer une invite contrôlant ton, format et longueur

Objectifs de la consigne : ton **amical et clair**, **puces**, **paraphrase** (pas de citation directe), **moins de 75 mots** au total.

> **Prompt (Étape 1) :**
>
> « Joue le rôle d'un **rédacteur en communication interne**. Réécris le paragraphe de politique ci-dessous pour une newsletter destinée aux employés.
>
> Exigences :
> - Adopte un **ton amical, clair et accessible** (public non technique).
> - Organise l'information sous forme de **puces**.
> - **Paraphrase** entièrement le texte : n'utilise aucune citation directe.
> - **Total inférieur à 75 mots.**
> - Ne conserve que les informations présentes dans le texte source.
>
> Texte source :
> "Les employés doivent s'assurer que tout accès distant aux systèmes internes est établi via le VPN sécurisé approuvé. En aucun cas, des connexions non sécurisées ou des appareils personnels dépourvus de protection des terminaux ne doivent être utilisés pour accéder à des données confidentielles ou à des communications sensibles." »

### Exemple de sortie générée

> **🔒 Rester connecté en toute sécurité**
> - Connecte-toi toujours aux systèmes internes via le **VPN approuvé** de l'entreprise.
> - Évite les connexions non sécurisées pour accéder à des données sensibles.
> - N'utilise pas d'appareils personnels sans protection des terminaux pour le travail confidentiel.

Vérifions la contrainte de longueur :

In [ ]:
sortie_etape1 = (
    "Rester connecté en toute sécurité "
    "Connecte-toi toujours aux systèmes internes via le VPN approuvé de l'entreprise. "
    "Évite les connexions non sécurisées pour accéder à des données sensibles. "
    "N'utilise pas d'appareils personnels sans protection des terminaux pour le travail confidentiel."
)

nb_mots = compter_mots(sortie_etape1)
print(f"Nombre de mots : {nb_mots}")
print("Respecte la limite (< 75 mots) :", nb_mots < 75)

## Étape 2 : Évaluer les résultats

Évaluation de la sortie de l'Étape 1 selon la grille fournie.

| Critère | Question directrice | Évaluation |
|---|---|---|
| **Pertinence** | Fidèle au message source ? | ✅ Oui — VPN approuvé, pas de connexions non sécurisées, pas d'appareils personnels non protégés. Les trois idées clés sont conservées. |
| **Clarté** | Plus facile pour un public non technique ? | ✅ Oui — phrases courtes, verbes d'action, vocabulaire simplifié (« Connecte-toi toujours… »). |
| **Structure** | Puces claires ? | ✅ Oui — trois puces distinctes, une idée par puce, avec un titre accrocheur. |
| **Ton** | Approprié pour une communication interne ? | ✅ Oui — amical, direct, tutoiement engageant sans être familier. |
| **Longueur** | ≤ 75 mots ? | ✅ Oui — bien en dessous de 75 mots (voir cellule de comptage). |
| **Exactitude factuelle** | Détails inventés ou supprimés ? | ✅ Aucun ajout ; les trois exigences de sécurité sont préservées sans invention. |

**Verdict :** la sortie satisfait les six critères. Le principal risque à surveiller reste l'**ajout d'éléments non présents** dans la source (technologies, règles supplémentaires), traité à l'Étape 3.

## Étape 3 : Détecter et atténuer les hallucinations

**Risque typique :** le modèle enrichit la sortie avec des éléments absents du texte — par exemple « activez l'authentification à deux facteurs », « utilisez un mot de passe fort » ou une techno nommée (Cisco AnyConnect, MFA…). Ces ajouts semblent plausibles mais **ne figurent pas** dans la politique source et constituent des hallucinations.

### Prompt révisé (strictement limité à l'entrée)

> **Prompt révisé (Étape 3) :**
>
> « Joue le rôle d'un rédacteur en communication interne. Réécris le paragraphe de politique ci-dessous pour une newsletter destinée aux employés.
>
> Exigences :
> - Ton amical et clair, public non technique.
> - Information sous forme de **puces**.
> - **Paraphrase uniquement le contenu du texte source. N'ajoute aucune nouvelle recommandation, règle, outil ou technologie qui ne figure pas explicitement dans le texte.**
> - N'omets aucune des exigences de sécurité présentes dans la source.
> - **Total inférieur à 75 mots.**
> - Si une information n'est pas dans le texte, ne l'invente pas.
>
> Texte source :
> "Les employés doivent s'assurer que tout accès distant… communications sensibles." »

**Ce que la révision ajoute :** une **clôture explicite du périmètre** (« paraphrase uniquement le contenu du texte source »), une **interdiction d'ajout** (nouvelles règles/outils/technologies), et une **règle anti-omission** pour ne pas perdre de détail critique. Ce cadrage réduit fortement le risque d'hallucination.

## Étape 4 : Paraphrase approfondie (public : stagiaires juniors)

Nouvelle consigne visant des **stagiaires juniors** : langage simple, phrases courtes, **maximum 4 puces**, **aucun jargon** d'entreprise ou juridique, ton **encourageant et informatif**.

> **Prompt (Étape 4) :**
>
> « Tu expliques une règle de sécurité informatique à de **nouveaux stagiaires** qui débutent dans l'entreprise. Reformule le paragraphe ci-dessous pour qu'il soit très facile à comprendre.
>
> Exigences :
> - **Langage simple et phrases courtes.**
> - **4 puces maximum.**
> - **Aucun jargon** d'entreprise ou juridique (explique les termes techniques avec des mots du quotidien).
> - Ton **encourageant et informatif**, comme si tu accompagnais un débutant.
> - Reste fidèle au texte source, sans rien ajouter.
>
> Texte source :
> "Les employés doivent s'assurer que tout accès distant… communications sensibles." »

### Exemple de sortie générée

> **Bienvenue ! Voici comment te connecter en toute sécurité 👇**
> - Pour te connecter au travail à distance, passe toujours par le **VPN de l'entreprise** (un outil qui protège ta connexion).
> - Ne te connecte pas depuis un réseau public ou non protégé.
> - Utilise seulement un appareil sécurisé par l'entreprise pour les infos importantes.
> - En cas de doute, demande à ton équipe : on est là pour t'aider !

*Le terme « VPN » est explicité, « protection des terminaux » devient « appareil sécurisé », et le ton reste rassurant. Note : la dernière puce (« demande à ton équipe ») est un ajout de ton — pour une fidélité stricte à la source, on peut la retirer conformément à l'Étape 3.*

## Étape 5 : Variante d'extraction de citation

Parfois, **citer** vaut mieux que **paraphraser**. Consigne extrayant la citation directe qui résume le mieux la politique.

> **Prompt (Étape 5) :**
>
> « À partir du texte de politique ci-dessous, **extrais une seule citation directe, mot pour mot**, qui résume le mieux l'exigence de sécurité essentielle. Ne modifie pas le texte cité et place-le entre guillemets. N'ajoute aucun commentaire.
>
> Texte source :
> "Les employés doivent s'assurer que tout accès distant… communications sensibles." »

### Citation extraite (exemple)

> « Les employés doivent s'assurer que tout accès distant aux systèmes internes est établi via le VPN sécurisé approuvé. »

Cette phrase capture l'obligation centrale de la politique (accès distant → uniquement via le VPN approuvé).

### Réponses aux questions

**Dans quel type de communication interne la citation est-elle plus appropriée que la paraphrase ?**

La citation directe est préférable lorsque la **formulation exacte a une valeur officielle ou contraignante** : documents de conformité, rappels de politique de sécurité, communications juridiques ou RH, avenants au règlement intérieur, ou tout message où l'employé doit connaître le **texte exact** de la règle. Citer garantit qu'aucune nuance n'est altérée et fait autorité (« voici la règle, telle qu'écrite »).

**Dans quelles circonstances les citations peuvent-elles présenter un risque ?**

- **Manque de clarté :** une citation dense ou jargonneuse peut rester incompréhensible pour un public non technique — l'inverse de l'objectif de micro-apprentissage.
- **Perte de contexte :** extraite seule, une phrase peut être mal interprétée ou sembler incomplète.
- **Obsolescence :** si la politique change, une citation copiée à plusieurs endroits devient fausse et doit être mise à jour partout.
- **Citation inexacte / hallucination :** un modèle peut altérer subtilement les mots en croyant citer — il faut donc toujours vérifier la citation par rapport à la source originale.

## Conclusion

Ce cycle complet illustre le **contrôle précis** de la génération :

- **Étape 1** — maîtrise simultanée du ton, du format (puces), de la méthode (paraphrase) et de la longueur (< 75 mots).
- **Étape 2** — évaluation systématique via une grille (pertinence, clarté, structure, ton, longueur, exactitude).
- **Étape 3** — cadrage anti-hallucination : limiter strictement le modèle à l'entrée.
- **Étape 4** — adaptation au public (stagiaires) par simplification du langage et de la structure.
- **Étape 5** — choix stratégique entre **citer** (fidélité au texte officiel) et **paraphraser** (accessibilité), avec conscience des risques.

**Principe clé :** un bon prompt fixe à l'avance tout ce qui peut l'être (rôle, format, longueur, périmètre) pour ne laisser au modèle que la reformulation utile — et l'on vérifie toujours la sortie plutôt que de la présumer correcte.